In [1]:
import logging
import os
import re
from collections import defaultdict

import yaml

# remove old log file
log_file = 'check_preprocessing.log'
if os.path.exists(log_file):
    os.remove(log_file)
logging.basicConfig(filename=log_file, level=logging.INFO, format='%(message)s', force=True)

datasplit_path = '../usleep_split_fixed.yaml'
prprocessing_log_path = '../logs/preprocess/2025-02-12_11-08-46/preprocess.log'

preprocess_log_recording_regex = r'.*processing recording (.*)'
preprocess_log_dataset_regex = r'.*SleepStudyDataset\(identifier: (.*), N pairs: .*'

with open(datasplit_path, 'r') as f:
    datasplit = yaml.safe_load(f)
    datasplit_recordings = defaultdict(list)
    for split, datasets in datasplit.items():
        for dataset, recordings in datasets.items():
            for recording in recordings:
                datasplit_recordings[dataset].append(recording)

with open(prprocessing_log_path, 'r') as f:
    preprocessing_log_lines = f.readlines()
    preprocessing_log_recordings = defaultdict(list)
    datasets = [(i, re.match(preprocess_log_dataset_regex, line).group(1))
                for i, line in enumerate(preprocessing_log_lines)
                if re.match(preprocess_log_dataset_regex, line)]
    for i in range(len(datasets)):
        start = datasets[i][0]
        if i == len(datasets) - 1:
            end = len(preprocessing_log_lines)
        else:
            end = datasets[i + 1][0]
        dataset = datasets[i][1]
        for line in preprocessing_log_lines[start:end]:
            if re.match(preprocess_log_recording_regex, line):
                recording = re.match(preprocess_log_recording_regex, line).group(1)
                preprocessing_log_recordings[dataset].append(recording)

dataset_intersect = set(datasplit_recordings.keys()).intersection(set(preprocessing_log_recordings.keys()))
if len(dataset_intersect) == len(datasplit_recordings) == len(preprocessing_log_recordings):
    logging.info('All datasets are present in both datasplit and preprocessing log.')
else:
    logging.info('Datasets missing in datasplit:')
    logging.info(set(datasplit_recordings.keys()).difference(dataset_intersect))
    logging.info('Datasets missing in preprocessing log:')
    logging.info(set(preprocessing_log_recordings.keys()).difference(dataset_intersect))
logging.info('')

for dataset in sorted(dataset_intersect):
    logging.info(f'Checking dataset {dataset}')
    recording_intersect = set(datasplit_recordings[dataset]).intersection(set(preprocessing_log_recordings[dataset]))
    if len(recording_intersect) == len(datasplit_recordings[dataset]) == len(preprocessing_log_recordings[dataset]):
        logging.info('All recordings are present in both datasplit and preprocessing log.')
    else:
        logging.info('Recordings missing in datasplit:')
        logging.info(set(datasplit_recordings[dataset]).difference(recording_intersect))
        logging.info('Recordings missing in preprocessing log:')
        logging.info(set(preprocessing_log_recordings[dataset]).difference(recording_intersect))
    logging.info('')